In [4]:
import logging
import sys
import pandas as pd

# Set up standard logging per capstone guidelines
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)


class TravelTimingRecommender:
    """Travel Timing Recommendation System for single corridor mobility[cite: 1]."""

    def __init__(self, data_path="Metro_Interstate_Traffic_Volume.csv"):
        logger.info(
            f"Initializing Recommendation Engine with dataset '{data_path}'..."
        )
        self.df = pd.read_csv(data_path)
        self._preprocess()

    def _preprocess(self):
        """Prepare datetime, day type, and congestion metrics."""
        self.df["date_time"] = pd.to_datetime(self.df["date_time"])
        self.df["hour"] = self.df["date_time"].dt.hour
        self.df["day_of_week"] = self.df["date_time"].dt.dayofweek
        self.df["is_weekend"] = self.df["day_of_week"].isin([5, 6])
        self.df["day_type"] = self.df["is_weekend"].map(
            {False: "weekday", True: "weekend"}
        )

        # Quartile-based congestion categories[cite: 1]
        q1, q2, q3 = (
            self.df["traffic_volume"].quantile([0.25, 0.5, 0.75]).values
        )

        def bucket(v):
            if v <= q1:
                return "Low"
            elif v <= q2:
                return "Medium"
            elif v <= q3:
                return "High"
            return "Severe"

        self.df["congestion_category"] = self.df["traffic_volume"].apply(bucket)

        # Calculate average hourly traffic profile by day type
        self.hourly_profile = (
            self.df.groupby(["day_type", "hour"])["traffic_volume"]
            .agg(["mean", "std"])
            .reset_index()
        )
        logger.info("Hourly traffic profiles built successfully.")

    def recommend_travel_window(
        self,
        is_weekend=False,
        weather_condition="Clear",
        target_period="daytime",
    ):
        """Generate an optimal travel window recommendation based on user constraints[cite: 1]."""
        day_type_str = "weekend" if is_weekend else "weekday"
        filtered_profile = self.hourly_profile[
            self.hourly_profile["day_type"] == day_type_str
        ].copy()

        # Define hourly bounds based on target travel period
        if target_period == "morning":
            filtered_profile = filtered_profile[
                (filtered_profile["hour"] >= 6)
                & (filtered_profile["hour"] <= 11)
            ]
        elif target_period == "daytime":
            filtered_profile = filtered_profile[
                (filtered_profile["hour"] >= 10)
                & (filtered_profile["hour"] <= 16)
            ]
        elif target_period == "evening":
            filtered_profile = filtered_profile[
                (filtered_profile["hour"] >= 16)
                & (filtered_profile["hour"] <= 21)
            ]

        # Find the hour with the lowest historical mean volume in the target window[cite: 1]
        best_row = filtered_profile.sort_values(by="mean").iloc[0]
        best_start_hour = int(best_row["hour"])
        best_end_hour = (best_start_hour + 1) % 24
        avg_vol = int(best_row["mean"])

        # Format time strings into 12-hour AM/PM format
        start_str = f"{best_start_hour % 12 or 12}:00 {'AM' if best_start_hour < 12 else 'PM'}"
        end_str = f"{best_end_hour % 12 or 12}:00 {'AM' if best_end_hour < 12 else 'PM'}"

        # Evaluate weather impact advisory[cite: 1]
        severe_weather_list = [
            "Rain",
            "Snow",
            "Thunderstorm",
            "Squall",
            "Fog",
            "Mist",
        ]
        weather_advisory = ""
        if any(
            w.lower() in weather_condition.lower() for w in severe_weather_list
        ):
            weather_advisory = (
                f" Note: Due to expected {weather_condition}, travel times may increase. "
                f"Leaving early in the {start_str} window is recommended to avoid severe weather bottlenecks."
            )

        # Construct plain-language recommendation text[cite: 1]
        recommendation = (
            f"For a {day_type_str} journey during the {target_period}, consider travelling "
            f"between {start_str} and {end_str}, when historical traffic volumes are typically lower "
            f"(averaging ~{avg_vol:,} vehicles/hr).{weather_advisory}"
        )

        return recommendation


# Execution
recommender = TravelTimingRecommender("Metro_Interstate_Traffic_Volume.csv")
rec = recommender.recommend_travel_window(
    is_weekend=False, weather_condition="Clear", target_period="daytime"
)
print(f'\nRecommendation:\n  "{rec}"')

2026-10-05 18:34:40,512 - INFO - Initializing Recommendation Engine with dataset 'Metro_Interstate_Traffic_Volume.csv'...
2026-10-05 18:34:40,642 - INFO - Hourly traffic profiles built successfully.

Recommendation:
  "For a weekday journey during the daytime, consider travelling between 10:00 AM and 11:00 AM, when historical traffic volumes are typically lower (averaging ~4,378 vehicles/hr)."
